# Session 16 Lab: Fine-Tuning with LoRA
**Course:** Noob to AI Expert | **Track:** Expert

⚠️ **Requires GPU runtime!** In Colab: Runtime → Change runtime type → T4 GPU

Fine-tune a small LLM to always respond in bullet points using LoRA.

In [ ]:
!pip install transformers peft datasets accelerate bitsandbytes -q
print('✅ Ready')

In [ ]:
import json, anthropic, os
os.environ.setdefault('ANTHROPIC_API_KEY', 'your-key-here')
client = anthropic.Anthropic()

# Generate training data: responses should always be in 3 bullet points
def generate_training_pair(topic):
    reply = client.messages.create(
        model='claude-haiku-4-5-20251001', max_tokens=200,
        system='Always respond in exactly 3 bullet points. Each bullet starts with •. Be concise.',
        messages=[{'role': 'user', 'content': f'Explain {topic}'}]
    ).content[0].text
    return {
        'messages': [
            {'role': 'system', 'content': 'Always respond in exactly 3 bullet points starting with •.'},
            {'role': 'user', 'content': f'Explain {topic}'},
            {'role': 'assistant', 'content': reply}
        ]
    }

topics = ['neural networks', 'gradient descent', 'overfitting', 'transformers',
          'embeddings', 'RAG', 'fine-tuning', 'prompt engineering', 'AI agents', 'LLMs']

training_data = [generate_training_pair(t) for t in topics]
with open('train.jsonl', 'w') as f:
    for item in training_data:
        f.write(json.dumps(item) + '\n')

print(f'Generated {len(training_data)} training examples')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

MODEL = 'microsoft/phi-2'  # Small but capable, runs on T4

print('Loading model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb_config, device_map='auto')

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=['q_proj', 'v_proj']
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()